# Giggsdance — MiniMax H3 at 60 fps

You are inside Modal, so **you are already authenticated. There is no token to paste.**

Set this notebook's own hardware to **CPU** (toolbar above). The heavy work runs in separate Modal functions on a B200; a GPU attached to *this* notebook would idle and still bill.

**Run Cell 1 first.** Every later cell depends on it. If a cell reports `No such file or directory`, Cell 1 has not run in this session — go back and run it.

### Check your credits before spending

If the dashboard banner says you have about **$1** of free credits, read **"Budget reality"** at the bottom before running cells 4 or 5. Self-hosting H3 needs ~90 GB downloaded and ~124 GB loaded into a GPU *before a single frame exists*, and that does not fit in $1.

### Licence

The [MiniMax H3 licence](https://github.com/Hvkki/minimax/blob/main/NOTICE.md) grants **no rights in the EU, UK, South Korea or USA**, and covers the model's **outputs**, not only its weights. Mark anything you publish as AI-generated.

## 1. Setup — run this first

Clones the repo, sets the working directory, and verifies every file is present. Re-runnable.

In [ ]:
# Cell 1 -- run this FIRST, and re-run it if a later cell says "No such file".
# Safe to run repeatedly: it clones once, then pulls.
import os, shutil, subprocess, sys

REPO = "/root/minimax"
URL = "https://github.com/Hvkki/minimax.git"

if shutil.which("git") is None:
    raise SystemExit("git is not installed in this image")

if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--quiet"], check=False)
    print(f"updated existing clone at {REPO}")
else:
    shutil.rmtree(REPO, ignore_errors=True)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", URL, REPO],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise SystemExit(
            "clone failed:\n" + (result.stderr or result.stdout)
            + "\n\nNo network? Modal notebooks have internet, but a proxy or a "
              "private repo would fail here."
        )
    print(f"cloned to {REPO}")

# chdir the kernel so every later `!` cell inherits it, and put the repo on
# sys.path so `import giggsdance` works in Python cells too.
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=False)

print("cwd:", os.getcwd())
missing = []
for name in ("doctor.py", "run.py", "notebook.py", "giggsdance/__init__.py"):
    ok = os.path.exists(name)
    print(f"  [{'OK' if ok else 'MISSING'}] {name}")
    if not ok:
        missing.append(name)

if missing:
    raise SystemExit(f"repo incomplete: {missing}")
print("\nSetup complete. Run the cells below in order.")


## 2. Free check — costs nothing

Validates interpolation, geometry and encoding on synthetic frames: frame counts, fps, bit depth, colour tags, A/V sync. No GPU, no weights, no charge.

In [ ]:
!cd /root/minimax && python run.py --dry-run

## 3. Cheap check — no GPU

Local environment, weight download and the unit suite run **inside** the container. Skips the expensive GPU probe.

**The ~90 GB weight download happens here**, on cheap CPU. One time; later runs skip it.

In [ ]:
!cd /root/minimax && modal run doctor.py --skip-gpu

## 4. Full check — boots a B200, loads H3

Loads ~124 GB and compares our generation call against the pipeline's real signature. This de-risks `stages/generate.py`, the only module in the repo its author could never execute.

**Skip this if you are on $1 of credits.**

In [ ]:
!cd /root/minimax && modal run doctor.py

## 5. Render

Cheap defaults: 5 s, `native` (no super-resolution — it measured at ~84% of post-processing time), 60 fps, 8 steps. `--budget-usd` becomes a hard container timeout, so an overrun is killed rather than billed.

In [ ]:
!cd /root/minimax && modal run run.py --budget-usd 1.0

## 6. Watch it

In [ ]:
from pathlib import Path
from IPython.display import Video, display

clips = sorted(Path("/root/minimax").glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if clips:
    print(clips[-1], f"{clips[-1].stat().st_size / 1e6:.1f} MB")
    display(Video(str(clips[-1]), embed=True, width=720))
else:
    print("nothing rendered yet -- run cell 5")

## Budget reality

Modal's free tier is **$30/month, but only ~$1 is available until a payment method is added**.

| Cost driver | Scale |
|---|---|
| Weight download | ~90 GB, CPU, one time |
| Volume storage | ~90 GB, ongoing |
| Model load | ~124 GB into GPU memory, **every cold start, before any frame exists** |
| Render | on top of all the above |

The model load alone can consume most of a dollar. With the full **$30** unlocked this notebook is comfortable — a 5 s native clip is a small fraction of it.

**On $1 and not adding a card?** Self-hosting is the wrong tool. Use [MiniMax's hosted API](https://platform.minimax.io/): no weights, no 124 GB load, no storage, and it is **globally available** so the territory restriction does not apply. It also includes `H3-Context-IR` and native 2K via `H3-Regenerate-2K`, neither of which is open source — self-hosting cannot give you them at all. This repo's prompt builder, 60 fps conversion, upscaling and encoding all still apply to API output.

## Troubleshooting

| Symptom | Fix |
|---|---|
| `FileNotFoundError` / `No such file` from `modal run` | Cell 1 has not run in this session — run it |
| `Bad Request: Unsupported URL` when importing | you pasted the **repo** URL; Import from URL needs a `.ipynb` link |
| `SyntaxError: invalid syntax` | shell command in a Python cell — add `!` |
| `ModuleNotFoundError: giggsdance` | re-run Cell 1 |
| timeout | raise `--budget-usd`, or lower `--steps` |
| generation fails | `!cd /root/minimax && modal run run.py --describe` |